In [1]:
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_sql_query_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
from urllib.parse import quote_plus

In [4]:
host = 'localhost'
port = '3306'
username = 'root'
password = '963Jinu@*'
database_schema = 'text_to_sql'

password_encoded = quote_plus(password)

mysql_uri = f"mysql+pymysql://{username}:{password_encoded}@{host}:{port}/{database_schema}"

db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

In [5]:
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=1)

db.get_table_info()

'\nCREATE TABLE `2017_budgets` (\n\t`Product Name` TEXT, \n\t`2017 Budgets` DOUBLE\n)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB\n\n/*\n1 rows from 2017_budgets table:\nProduct Name\t2017 Budgets\nProduct 1\t3016489.2089999998\n*/\n\n\nCREATE TABLE customers (\n\t`Customer Index` INTEGER, \n\t`Customer Names` TEXT\n)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB\n\n/*\n1 rows from customers table:\nCustomer Index\tCustomer Names\n1\tGeiss Company\n*/\n\n\nCREATE TABLE products (\n\t`Index` INTEGER, \n\t`Product Name` TEXT\n)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB\n\n/*\n1 rows from products table:\nIndex\tProduct Name\n1\tProduct 1\n*/\n\n\nCREATE TABLE regions (\n\tid INTEGER, \n\tname TEXT, \n\tcounty TEXT, \n\tstate_code TEXT, \n\tstate TEXT, \n\ttype TEXT, \n\tlatitude DOUBLE, \n\tlongitude DOUBLE, \n\tarea_code INTEGER, \n\tpopulation INTEGER, \n\thouseholds INTEGER, \n\tmedian_income INTEGER, \n\tland_area INTEGER, \

In [6]:
# Create the LLM Prompt Template
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide the sql query dont include anything else. Provide me sql query in a single line dont add line breaks
Table Schema: {schema}
Question: {question}
SQL Query:
"""

prompt = ChatPromptTemplate.from_template(template)

In [7]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [8]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    api_key='xyz'
)

In [9]:
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [10]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GoogleModelNotFoundError: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}

In [11]:
llm = ChatGoogleGenerativeAI(
    model='gemini-3.6-flash',
    api_key='xyz'
)

In [12]:
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [13]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


In [14]:
import re

query = re.search(r"```sql\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)

if query:
    query=query.group(1).strip()

In [15]:
db.run(query)

TypeError: Query expression has unknown type: <class 'NoneType'>

In [16]:
import re

query = re.sub(r"```sql|```", "", resp, flags=re.IGNORECASE).strip()

print(query)

db.run(query)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


'[(1356976.996,)]'

In [17]:
import re

query = re.sub(r"```sql|```", "", resp, flags=re.IGNORECASE).strip()

print(query)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


In [18]:
db.run(query)

'[(1356976.996,)]'

### RAGAS


In [19]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [20]:
%pip install ragas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
%pip install ragas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [5]:
%pip install langchain-google-vertexai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install langchain-google-vertexai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
context = db.get_table_info()

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [19]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [20]:
%pip uninstall -y ragas

Found existing installation: ragas 0.4.3Note: you may need to restart the kernel to use updated packages.

Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3


In [21]:
%pip install ragas==0.3.9

                                              0.0/366.7 kB ? eta -:--:--
     ------                                  61.4/366.7 kB 1.1 MB/s eta 0:00:01
     ---------------                        153.6/366.7 kB 1.5 MB/s eta 0:00:01
     -------------------------------------  358.4/366.7 kB 2.5 MB/s eta 0:00:01
     -------------------------------------- 366.7/366.7 kB 2.5 MB/s eta 0:00:00
                                              0.0/222.8 kB ? eta -:--:--
     ------------------------------------- 222.8/222.8 kB 14.2 MB/s eta 0:00:00
                                              0.0/62.8 kB ? eta -:--:--
     ---------------------------------------- 62.8/62.8 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [2]:
%pip show ragas

Name: ragasNote: you may need to restart the kernel to use updated packages.

Version: 0.3.9
Summary: Evaluation framework for RAG and LLM applications
Home-page: 
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or managemen

In [3]:
import ragas
print(ragas.__version__)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [4]:
%pip install "langchain-community==0.4.1"

                                              0.0/2.5 MB ? eta -:--:--
                                              0.0/2.5 MB 653.6 kB/s eta 0:00:04
     -                                        0.1/2.5 MB 1.2 MB/s eta 0:00:03
     ----                                     0.3/2.5 MB 1.8 MB/s eta 0:00:02
     --------                                 0.5/2.5 MB 2.8 MB/s eta 0:00:01
     -----------------                        1.1/2.5 MB 4.6 MB/s eta 0:00:01
     ---------------------------              1.7/2.5 MB 6.1 MB/s eta 0:00:01
     -----------------------------------      2.3/2.5 MB 6.9 MB/s eta 0:00:01
     ---------------------------------------- 2.5/2.5 MB 7.0 MB/s eta 0:00:00
                                              0.0/51.0 kB ? eta -:--:--
     ---------------------------------------- 51.0/51.0 kB ? eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.4.2
    Uninstalling langchain-community-0.4.2:
      Succe


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ImportError: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet


In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [10]:
%pip install langchain-groq

                                              0.0/137.5 kB ? eta -:--:--
     --------                                30.7/137.5 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 137.5/137.5 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.2.0
    Uninstalling groq-1.2.0:
      Successfully uninstalled groq-1.2.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    api_key="xyz"
)

In [2]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

ModuleNotFoundError: No module named 'langchain_huggingface'

In [3]:
%pip install langchain-huggingface

                                              0.0/2.9 MB ? eta -:--:--
                                              0.0/2.9 MB 1.9 MB/s eta 0:00:02
     --                                       0.2/2.9 MB 2.1 MB/s eta 0:00:02
     -----                                    0.4/2.9 MB 2.9 MB/s eta 0:00:01
     --------                                 0.6/2.9 MB 3.5 MB/s eta 0:00:01
     --------------                           1.1/2.9 MB 5.6 MB/s eta 0:00:01
     ----------------------                   1.6/2.9 MB 6.7 MB/s eta 0:00:01
     -----------------------------            2.1/2.9 MB 7.5 MB/s eta 0:00:01
     -------------------------------------    2.7/2.9 MB 8.1 MB/s eta 0:00:01
     ---------------------------------------- 2.9/2.9 MB 7.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [2]:
%pip install sentence-transformers

                                              0.0/740.6 kB ? eta -:--:--
     -                                        30.7/740.6 kB ? eta -:--:--
     -----                                  112.6/740.6 kB 1.3 MB/s eta 0:00:01
     ------------                           245.8/740.6 kB 1.9 MB/s eta 0:00:01
     --------------------------             522.2/740.6 kB 2.7 MB/s eta 0:00:01
     -------------------------------------- 740.6/740.6 kB 3.3 MB/s eta 0:00:00
                                              0.0/12.3 MB ? eta -:--:--
     -                                        0.5/12.3 MB 16.2 MB/s eta 0:00:01
     ---                                      1.2/12.3 MB 12.7 MB/s eta 0:00:01
     -----                                    1.6/12.3 MB 11.1 MB/s eta 0:00:01
     ------                                   2.1/12.3 MB 11.2 MB/s eta 0:00:01
     --------                                 2.7/12.3 MB 11.3 MB/s eta 0:00:01
     ----------                               3.1/12.3 MB 11.


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

NameError: name 'LangchainLLMWrapper' is not defined

In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [4]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

NameError: name 'llm' is not defined

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="gemma2-9b-it",
    api_key="xyz"
)

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [7]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\dell\AppData\Local\Temp\ipykernel_924\108186089.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
C:\Users\dell\AppData\Local\Temp\ipykernel_924\108186089.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [9]:
from ragas import evaluate
from ragas.metrics import AspectCritic, RubricsScore

maliciousness_definition = (
    "Is the response intended to harm, deceive, or exploit users?"
)

aspect_critic = AspectCritic(
    name="maliciousness",
    definition=maliciousness_definition,
    llm=evaluator_llm,
)

# adapeted google's helpfulness_prompt_template
helpfulness_rubrics = {
    "score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.",
    "score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.",
    "score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.",
    "score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs.  It may include additional insights that go slightly beyond the immediate instruction.  The user would feel quite satisfied with the content in the response.",
    "score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.",
}

rubrics_score = RubricsScore(name="helpfulness", rubrics=helpfulness_rubrics, llm=evaluator_llm)

In [10]:
from ragas import evaluate
from ragas.metrics import ContextPrecision, Faithfulness

context_precision = ContextPrecision(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

In [11]:
retrieved_contexts = [context]

NameError: name 'context' is not defined

In [24]:
context = db.get_table_info()

In [25]:
retrieved_contexts = [context]

In [32]:
import re

user_inputs = [
    "What was the budget of Product 12",
    "What are the names of all products in the products table?",
    "List all customer names from the customers table.",
    "Find the name and state of all regions in the regions table.",
    "What is the name of the customer with Customer Index = 1"
]

responses = []

for question in user_inputs:
    resp = sql_chain.invoke({"question": question})
    query = re.sub(r"```sql|```", "", resp, flags=re.IGNORECASE).strip()
    responses.append(query)

In [33]:
len(responses)

5

In [31]:
n = len(user_inputs)
samples = []

for i in range(n):
    sample = SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)

IndexError: list index out of range

In [27]:
references = [
    "SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';",
    "SELECT `Product Name` FROM products;",
    "SELECT `Customer Names` FROM customers;",
    "SELECT name, state FROM regions;",
    "SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;"
]

In [28]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

In [29]:
n = len(user_inputs)
samples = []

for i in range(n):
    sample = SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)

IndexError: list index out of range

In [34]:
n = len(user_inputs)
samples = []

for i in range(n):
    sample = SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)

In [35]:
ragas_eval_dataset = EvaluationDataset(samples=samples)
ragas_eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,What was the budget of Product 12,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...
1,What are the names of all products in the prod...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Product Name` FROM products;,SELECT `Product Name` FROM products;
2,List all customer names from the customers table.,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers;,SELECT `Customer Names` FROM customers;
3,Find the name and state of all regions in the ...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT name, state FROM regions;","SELECT name, state FROM regions;"
4,What is the name of the customer with Customer...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers WHERE `...,SELECT `Customer Names` FROM customers WHERE `...


In [36]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

NameError: name 'context_precision' is not defined

In [37]:
from ragas import evaluate
from ragas.metrics import ContextPrecision, Faithfulness

context_precision = ContextPrecision(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

NameError: name 'evaluator_llm' is not defined

In [38]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\dell\AppData\Local\Temp\ipykernel_17364\108186089.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)


NameError: name 'embeddings' is not defined

In [39]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [40]:
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\dell\AppData\Local\Temp\ipykernel_17364\1901526855.py:1: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [41]:
from ragas import evaluate
from ragas.metrics import ContextPrecision, Faithfulness

context_precision = ContextPrecision(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

In [42]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

NameError: name 'rubrics_score' is not defined

In [43]:
from ragas import evaluate
from ragas.metrics import AspectCritic, RubricsScore

maliciousness_definition = (
    "Is the response intended to harm, deceive, or exploit users?"
)

aspect_critic = AspectCritic(
    name="maliciousness",
    definition=maliciousness_definition,
    llm=evaluator_llm,
)

helpfulness_rubrics = {
    "score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.",
    "score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.",
    "score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.",
    "score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs. It may include additional insights that go slightly beyond the immediate instruction. The user would feel quite satisfied with the content in the response.",
    "score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.",
}

rubrics_score = RubricsScore(
    name="helpfulness",
    rubrics=helpfulness_rubrics,
    llm=evaluator_llm
)

In [44]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.
C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning:

{'context_precision': 1.0000, 'helpfulness': nan}

In [1]:
from langchain_groq import ChatGroq

groq_llm = ChatGroq(
    model="gemma2-9b-it",
    api_key="xyz"
)

In [2]:
evaluator_llm = LangchainLLMWrapper(groq_llm)

NameError: name 'LangchainLLMWrapper' is not defined

In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [4]:
evaluator_llm = LangchainLLMWrapper(groq_llm)

C:\Users\dell\AppData\Local\Temp\ipykernel_19236\2237830263.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(groq_llm)


In [5]:
from ragas.metrics import ContextPrecision, RubricsScore

context_precision = ContextPrecision(llm=evaluator_llm)

rubrics_score = RubricsScore(
    name="helpfulness",
    rubrics=helpfulness_rubrics,
    llm=evaluator_llm
)

NameError: name 'helpfulness_rubrics' is not defined

In [6]:
helpfulness_rubrics = {
    "score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.",
    "score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.",
    "score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.",
    "score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs. It may include additional insights that go slightly beyond the immediate instruction. The user would feel quite satisfied with the content in the response.",
    "score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.",
}

In [7]:
rubrics_score = RubricsScore(
    name="helpfulness",
    rubrics=helpfulness_rubrics,
    llm=evaluator_llm
)

In [8]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

NameError: name 'ragas_eval_dataset' is not defined

In [9]:
ragas_eval_dataset = EvaluationDataset(samples=samples)

NameError: name 'EvaluationDataset' is not defined

In [10]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

In [11]:
ragas_eval_dataset = EvaluationDataset(samples=samples)

NameError: name 'samples' is not defined

In [12]:
n = len(user_inputs)
samples = []

for i in range(n):
    sample = SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)

NameError: name 'user_inputs' is not defined

In [13]:
user_inputs = [
    "What was the budget of Product 12",
    "What are the names of all products in the products table?",
    "List all customer names from the customers table.",
    "Find the name and state of all regions in the regions table.",
    "What is the name of the customer with Customer Index = 1"
]

In [14]:
references = [
    "SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';",
    "SELECT `Product Name` FROM products;",
    "SELECT `Customer Names` FROM customers;",
    "SELECT name, state FROM regions;",
    "SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;"
]

In [15]:
print("sql_chain:", "sql_chain" in globals())
print("responses:", "responses" in globals())
print("retrieved_contexts:", "retrieved_contexts" in globals())
print("samples:", "samples" in globals())

sql_chain: False
responses: False
retrieved_contexts: False
samples: False


In [16]:
print("db:", "db" in globals())
print("llm:", "llm" in globals())
print("prompt:", "prompt" in globals())
print("get_schema:", "get_schema" in globals())

db: False
llm: False
prompt: False
get_schema: False


In [17]:
from langchain_community.utilities import SQLDatabase
from urllib.parse import quote_plus

host = "localhost"
port = "3306"
username = "root"
password = "YOUR_MYSQL_PASSWORD"
database_schema = "text_to_sql"

password_encoded = quote_plus(password)

mysql_uri = f"mysql+pymysql://{username}:{password_encoded}@{host}:{port}/{database_schema}"

db = SQLDatabase.from_uri(
    mysql_uri,
    sample_rows_in_table_info=1
)

print("Database connected successfully")

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [18]:
from langchain_community.utilities import SQLDatabase
from urllib.parse import quote_plus

host = "localhost"
port = "3306"
username = "root"
password = "963Jinu@*"
database_schema = "text_to_sql"

password_encoded = quote_plus(password)

mysql_uri = f"mysql+pymysql://{username}:{password_encoded}@{host}:{port}/{database_schema}"

db = SQLDatabase.from_uri(
    mysql_uri,
    sample_rows_in_table_info=1
)

print("Database connected successfully")

Database connected successfully


In [19]:
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide the sql query dont include anything else. Provide me sql query in a single line dont add line breaks
Table Schema: {schema}
Question: {question}
SQL Query:
"""

prompt = ChatPromptTemplate.from_template(template)

In [20]:
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key="xyz"
)

In [22]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [23]:
import re

responses = []

for question in user_inputs:
    resp = sql_chain.invoke({"question": question})

    query = re.sub(
        r"```sql|```",
        "",
        resp,
        flags=re.IGNORECASE
    ).strip()

    responses.append(query)

print("Number of responses:", len(responses))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 217.602116ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

In [24]:
print("Number of responses:", len(responses))
print(responses)

Number of responses: 0
[]


In [25]:
from langchain_groq import ChatGroq

groq_sql_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="xyz"
)

print("Groq SQL LLM ready")

Groq SQL LLM ready


In [26]:
import re

responses = []

for question in user_inputs:
    resp = groq_sql_llm.invoke(
        f"""Based on the database schema below, write a SQL query that answers the user's question.

Table Schema:
{db.get_table_info()}

Question:
{question}

Only provide the SQL query. Do not include explanations or markdown."""
    )

    query = re.sub(
        r"```sql|```",
        "",
        resp.content,
        flags=re.IGNORECASE
    ).strip()

    responses.append(query)

print("Number of responses:", len(responses))

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [27]:
from langchain_groq import ChatGroq

groq_sql_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key="xyz"
)

print("Groq SQL LLM ready")

Groq SQL LLM ready


In [28]:
import re

responses = []

for question in user_inputs:
    resp = groq_sql_llm.invoke(
        f"""Based on the table schema below, write a SQL query that would answer the user's question.

Remember: Only provide the SQL query. Do not include anything else.

Table Schema:
{db.get_table_info()}

Question:
{question}

SQL Query:"""
    )

    query = re.sub(
        r"```sql|```",
        "",
        resp.content,
        flags=re.IGNORECASE
    ).strip()

    responses.append(query)

print("Number of responses:", len(responses))

Number of responses: 5


In [29]:
context = db.get_table_info()
retrieved_contexts = [context]

print("Retrieved context created")

Retrieved context created


In [30]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

samples = []

for i in range(len(user_inputs)):
    sample = SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)

ragas_eval_dataset = EvaluationDataset(samples=samples)

print("Ragas evaluation dataset created")

Ragas evaluation dataset created


In [31]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Exception raised in Job[9]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}})
Exception raised in Job[3]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}})
Exception raised in Job[2]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}})
Exception raised in Job[5]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}})
Exception raised in Job[1]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}})
Exception raised in Job[8]: AuthenticationError(Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invali

{'context_precision': nan, 'helpfulness': nan}

In [32]:
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(groq_sql_llm)

print("Ragas evaluator ready")

Ragas evaluator ready


C:\Users\dell\AppData\Local\Temp\ipykernel_19236\418204882.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(groq_sql_llm)


In [33]:
from ragas.metrics import ContextPrecision, RubricsScore

context_precision = ContextPrecision(
    llm=evaluator_llm
)

rubrics_score = RubricsScore(
    name="helpfulness",
    rubrics=helpfulness_rubrics,
    llm=evaluator_llm
)

print("Ragas metrics ready")

Ragas metrics ready


In [34]:
from ragas import evaluate

ragas_metrics = [context_precision, rubrics_score]

result = evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)

result

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'context_precision': 1.0000, 'helpfulness': 4.0000}